## Fraud Detection - Model
RandomForestClassifier with hyperparameter tuning to predict isFraud

In [1]:
# Load clean data. Given severe class imbalance (0.13% fraud) and 6.3M rows,
# we sample for training speed while keeping ALL fraud cases plus a random
# sample of non-fraud cases (documented decision, not hiding data issues)
import pandas as pd
import numpy as np

df = pd.read_csv("../data/fraud_transactions_clean.csv")

fraud = df[df['isFraud'] == 1]
non_fraud = df[df['isFraud'] == 0].sample(n=200000, random_state=42)
df_sample = pd.concat([fraud, non_fraud]).sample(frac=1, random_state=42).reset_index(drop=True)
print(df_sample.shape)
df_sample['isFraud'].value_counts()

(208213, 12)


isFraud
0    200000
1      8213
Name: count, dtype: int64

In [2]:
# Calculate fraud rate in sampled dataset
(df_sample['isFraud'].sum() / len(df_sample) * 100).round(3)

np.float64(3.945)

### Sampling Strategy
- Full dataset: 6.36M rows, severe imbalance (0.13% fraud)
- Sampled: all 8,213 fraud cases + random 200K non-fraud = 208,213 rows
- Fraud rate in sample: 3.945%
- Reason: computational speed for hyperparameter search within project timeline
- Still imbalanced — realistic for model training, not artificially balanced

### M1: Split the Data

In [3]:
# Train/test split - stratify to preserve fraud ratio in both sets
from sklearn.model_selection import train_test_split

X = df_sample.drop(columns=['isFraud'])
y = df_sample['isFraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(166570, 11) (41643, 11)


### Train/Test Split
- 80/20 split, stratified on isFraud to preserve fraud ratio in both sets
- random_state=42 for reproducibility

### M2: Fit Initial Model

In [4]:
# Fit baseline RandomForestClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     40000
           1       0.97      0.96      0.97      1643

    accuracy                           1.00     41643
   macro avg       0.98      0.98      0.98     41643
weighted avg       1.00      1.00      1.00     41643



### Baseline RandomForestClassifier
- Precision (fraud): 0.97 | Recall (fraud): 0.96 | F1 (fraud): 0.97
- Overall accuracy: 1.00 (expected given imbalance, but precision/recall on fraud class are the meaningful metrics here)
- Strong baseline before tuning

### M3: Hyperparameter Tuning

In [5]:
# Hyperparameter tuning with RandomizedSearchCV (faster than GridSearch given time constraints)
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)
print("Best params:", rf_search.best_params_)

Best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': 20}


In [6]:
# Retrain with best hyperparameters and evaluate
best_rf = rf_search.best_estimator_
y_pred_best = best_rf.predict(X_test)
print(classification_report(y_test, y_pred_best))

final_f1 = f1_score(y_test, y_pred_best)
print("Final F1:", final_f1)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     40000
           1       0.97      0.96      0.97      1643

    accuracy                           1.00     41643
   macro avg       0.99      0.98      0.98     41643
weighted avg       1.00      1.00      1.00     41643

Final F1: 0.9663402692778458


In [7]:
# Feature importance from tuned model
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances

balance_diff_orig    0.357140
amount               0.142348
newbalanceDest       0.139179
oldbalanceOrg        0.116428
step                 0.063121
newbalanceOrig       0.063096
oldbalanceDest       0.048970
type_TRANSFER        0.048825
type_CASH_OUT        0.014002
type_PAYMENT         0.006704
type_DEBIT           0.000186
dtype: float64

### Feature Importance
- `balance_diff_orig` (engineered): 35.7% — most important feature, confirms EDA finding that fraud drains accounts
- `amount`: 14.2%, `newbalanceDest`: 13.9%, `oldbalanceOrg`: 11.6%
- `type_DEBIT`: 0.02% — negligible, consistent with EDA showing 0% fraud in DEBIT transactions
- Validates our feature engineering: the engineered balance_diff feature outperforms all raw balance columns individually